In [1]:
from xgboost import XGBClassifier
# from xgboost import XGBRegressor

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

import matplotlib.pyplot as plt
import pandas as pd

import mlflow
import optuna  # Hyperpaameter Optimizer tune framework
from mlflow import MlflowClient

In [2]:

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Breast Cancer XGBoost")

print("Connected to : ", mlflow.get_tracking_uri())

2026/10/18 16:32:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/18 16:32:24 INFO mlflow.store.db.utils: Updating database tables
2026/10/18 16:32:26 INFO mlflow.tracking.fluent: Experiment with name 'Breast Cancer XGBoost' does not exist. Creating a new experiment.


Connected to :  sqlite:///mlflow.db


In [3]:
data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Classes: ",data.target_names)
print("Training : ",X_train.shape)
print("Validation : ",X_val.shape)

Classes:  ['malignant' 'benign']
Training :  (398, 30)
Validation :  (85, 30)


In [ ]:
def objective(trial):

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        params = {
            "max_depth":trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree":trial.suggest_float("colsample_bytree", 0.6, 1.0),

            "n_estimators":1000,
            "early_stopping_rounds":30,
            "eval_metric":"logloss",
            "random_state":42,
        }

        mlflow.log_params(params)

        